# Chinese world — Cultura vs Cross-Verified, what is the difference?

Take **all individuals who could plausibly belong to the Chinese world** in each database, and look at who is in one but not the other.

- **Cultura side** — every individual whose `polity_name` (semicolon list) contains any canonical Chinese dynasty (Shang → Qing).
- **Cross-Verified side** — every individual whose `citizenship_1_b` *or* `citizenship_2_b` falls into the **Chinese world citizenship set**, derived from Cultura's own `polities_modern_countries_cliopatria` table (the polities map to: People's Republic of China, Taiwan, North Korea, Mongolia, Russia — those that have CV equivalents are kept).

What we then show:
1. Total counts in each database, the Q-id intersection, and the two set differences (Cultura-only, CV-only).
2. Per-century counts and the difference per century.
3. A few example individuals on each side of the difference, so it's clear what kind of people each database is adding."

## 1. Configuration

In [ ]:
import duckdb
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

DB_PATH = '../data/humans_clean.duckdb'
CV_PATH = '../data/similar_databases/cross-verified-database/cross-verified-database.utf8.csv.gz'

CENTURY_MIN, CENTURY_MAX = -16, 20  # −1600 BCE → 2000 CE

COL_CULTURA = '#2f5b8a'
COL_CV      = '#b5542a'

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 12,
    'font.family': 'DejaVu Sans',
})

## 2. Chinese world definitions

Cultura: 25 canonical dynasties. CV: derived from Cultura's `polities_modern_countries_cliopatria` mapping plus CV-side aliases (`People's Republic of China` → `China`, etc.) — keeping only entries that exist in CV's citizenship vocabulary.

In [ ]:
CHINESE_DYNASTIES = [
    'Shang Dynasty', 'Zhou Dynasty', 'Qin Dynasty', 'Han Dynasty', 'Xin Dynasty',
    'Western Jin', 'Eastern Jin', 'Liu Song Dynasty', 'Liang Dynasty', 'Chen Dynasty',
    'Northern Wei', 'Eastern Wei', 'Western Wei', 'Northern Zhou', 'Northern Qi',
    'Sui Dynasty', 'Tang Dynasty', 'Five Dynasties and Ten Kingdoms',
    'Northern Song', 'Southern Song', 'Liao Dynasty', 'Western Xia',
    'Yuan Dynasty', 'Ming Dynasty', 'Qing Dynasty',
]

# Aliases — Cultura country_name → CV citizenship_1_b
CULTURA_TO_CV_ALIAS = {
    "People's Republic of China": 'China',
    'North Korea':                 'North_Korea',
}

def cultura_country_to_cv(name):
    if name is None:
        return None
    return CULTURA_TO_CV_ALIAS.get(name, name.replace(' ', '_'))

# Look up the modern-country mapping for the dynasties
con = duckdb.connect(DB_PATH, read_only=True)
ph = ','.join('?' * len(CHINESE_DYNASTIES))
country_table = con.execute(f"""
    SELECT DISTINCT polity_name, country_name
    FROM polities_modern_countries_cliopatria
    WHERE polity_name IN ({ph}) AND country_name IS NOT NULL
    """, CHINESE_DYNASTIES).pl()
country_table = country_table.with_columns(
    pl.col('country_name').map_elements(cultura_country_to_cv, return_dtype=pl.Utf8).alias('cv_citizenship')
)

# CV vocabulary — needed to drop translations CV does not have
cv_vocab = pl.read_csv(
    CV_PATH,
    columns=['citizenship_1_b', 'citizenship_2_b'],
)
CV_KNOWN = (
    set(cv_vocab.filter(pl.col('citizenship_1_b').is_not_null())['citizenship_1_b'].to_list())
    | set(cv_vocab.filter(pl.col('citizenship_2_b').is_not_null())['citizenship_2_b'].to_list())
)
del cv_vocab

country_table = country_table.with_columns(
    pl.col('cv_citizenship').is_in(CV_KNOWN).alias('in_cv_vocab')
)
CV_CHINESE_CITIZENSHIPS = set(
    country_table.filter(pl.col('in_cv_vocab'))['cv_citizenship'].to_list()
)
con.close()

print(f'Dynasties: {len(CHINESE_DYNASTIES)}')
print(f'Derived CV citizenships ({len(CV_CHINESE_CITIZENSHIPS)}): '
      f'{sorted(CV_CHINESE_CITIZENSHIPS)}')
country_table

## 3. Cultura — load Chinese individuals and assign a floruit year

Joins `individuals_cliopatria` (for the polity filter) with `individuals_floruit_period` (for `floruit_year`). Each individual is counted once.

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)

like_clauses = ' OR '.join(["';' || ic.polity_name || ';' LIKE ?"] * len(CHINESE_DYNASTIES))
params = [f'%;{d};%' for d in CHINESE_DYNASTIES]

cultura = conn.execute(f"""
    SELECT DISTINCT ic.wikidata_id,
           fp.floruit_year,
           fp.floruit_period_start AS s,
           fp.floruit_period_end   AS e
    FROM individuals_cliopatria ic
    JOIN individuals_floruit_period fp USING (wikidata_id)
    WHERE ({like_clauses})
""", params).pl().unique(subset=['wikidata_id'])
conn.close()

# Use floruit_year if present, else midpoint of [start, end]
cultura = cultura.with_columns(
    pl.coalesce(
        pl.col('floruit_year'),
        ((pl.col('s') + pl.col('e')) / 2).round(0).cast(pl.Int64),
    ).alias('fy')
).filter(pl.col('fy').is_not_null()).with_columns(
    pl.col('fy').cast(pl.Int64),
    (pl.col('fy') // 100).alias('century').cast(pl.Int64),
)

n_cultura = cultura.height
print(f'Cultura Chinese-world individuals : {n_cultura:,}')

## 4. Cross-Verified — load Chinese individuals and assign a floruit year

Filter on `citizenship_1_b` or `citizenship_2_b`. Floruit year:
- if `birth` and `death` are both known → midpoint;
- if only `birth` → `birth + 30`;
- if only `death` → `death − 30`.

The same `+30` adult-onset convention is what the original Pantheon paper uses.

In [ ]:
cv = pl.read_csv(
    CV_PATH,
    columns=['wikidata_code', 'name', 'citizenship_1_b', 'citizenship_2_b',
             'birth', 'death', 'updated_death_date',
             'level1_main_occ', 'level2_main_occ', 'level3_main_occ',
             'list_wikipedia_editions'],
)

cv = cv.filter(
    pl.col('citizenship_1_b').is_in(CV_CHINESE_CITIZENSHIPS)
    | pl.col('citizenship_2_b').is_in(CV_CHINESE_CITIZENSHIPS)
).with_columns(
    pl.coalesce([pl.col('death'), pl.col('updated_death_date')]).alias('death_eff'),
).with_columns(
    pl.when(pl.col('birth').is_not_null() & pl.col('death_eff').is_not_null())
      .then((pl.col('birth') + pl.col('death_eff')) / 2)
    .when(pl.col('birth').is_not_null())
      .then(pl.col('birth') + 30)
    .when(pl.col('death_eff').is_not_null())
      .then(pl.col('death_eff') - 30)
    .otherwise(None)
    .alias('fy')
).filter(pl.col('fy').is_not_null()).unique(subset=['wikidata_code']).with_columns(
    pl.col('fy').round(0).cast(pl.Int64),
).with_columns(
    (pl.col('fy') // 100).alias('century').cast(pl.Int64),
)

n_cv = cv.height
print(f'Cross-Verified Chinese-world individuals : {n_cv:,}')

## 5. Total counts (overall and overlap)

In [ ]:
cultura_qids = set(cultura['wikidata_id'].to_list())
cv_qids      = set(cv.filter(pl.col('wikidata_code').is_not_null())['wikidata_code'].cast(pl.Utf8).to_list())

in_both = cultura_qids & cv_qids
cultura_only = cultura_qids - cv_qids
cv_only = cv_qids - cultura_qids

summary = pl.DataFrame({
    'metric': [
        'Cultura (Chinese polities)',
        'Cross-Verified (Chinese citizenships)',
        'Shared (Q-id overlap)',
        'Cultura only',
        'Cross-Verified only',
    ],
    'count': [n_cultura, n_cv, len(in_both), len(cultura_only), len(cv_only)],
})
summary

## 6. Where do the two databases differ?

For each individual on each side, we have a name, century, and main occupation (CV side) / `name_en` from `individuals_floruit_period` (Cultura side). Look at the *kind* of people present in one but not the other."

In [ ]:
# Enrich Cultura individuals with names
con = duckdb.connect(DB_PATH, read_only=True)
cultura_meta = con.execute("SELECT wikidata_id, name_en FROM individuals WHERE wikidata_id IN ({})".format(
        ','.join(['?'] * len(cultura_qids))
    ), list(cultura_qids)).pl()
con.close()
cultura_full = cultura.join(cultura_meta, on='wikidata_id', how='left')

# CV-only and Cultura-only views
cv_only_df      = cv.filter(pl.col('wikidata_code').is_in(cv_only))
cultura_only_df = cultura_full.filter(pl.col('wikidata_id').is_in(cultura_only))

print(f'Cultura-only        : {cultura_only_df.height:,}')
print(f'Cross-Verified-only : {cv_only_df.height:,}')

### 6.1 Difference per century — bar chart of unique additions

For each century, how many individuals does Cultura add that CV does not have, and vice-versa.

In [ ]:
centuries_arr = np.arange(CENTURY_MIN, CENTURY_MAX + 1)

def per_cent_counts(df_pl):
    counts = df_pl.group_by('century').agg(pl.len().alias('n'))
    by_c = dict(counts.iter_rows())
    return np.array([by_c.get(int(c), 0) for c in centuries_arr])

cu_only_per_c = per_cent_counts(cultura_only_df)
cv_only_per_c = per_cent_counts(cv_only_df)

fig, ax = plt.subplots(figsize=(11, 4.6))
w = 0.42
ax.bar(centuries_arr - w/2, cu_only_per_c, width=w,
       color=COL_CULTURA, label=f'Cultura only  ({cultura_only_df.height:,})')
ax.bar(centuries_arr + w/2, cv_only_per_c, width=w,
       color=COL_CV,      label=f'Cross-Verified only  ({cv_only_df.height:,})')
ax.set_xlabel('Century (year // 100, BCE ← → CE)')
ax.set_ylabel('Individuals exclusive to one DB')
ax.set_title('Chinese world — what each database adds, per century', loc='left', pad=12)
ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
ax.legend(loc='upper left', frameon=False)
fig.tight_layout()
plt.show()

### 6.2 What kind of people does Cross-Verified contribute that Cultura misses?

Distribution of `level1_main_occ` for the CV-only set.

In [ ]:
occ_dist = (
    cv_only_df.with_columns(pl.col('level1_main_occ').fill_null('(unknown)'))
    .group_by('level1_main_occ').agg(pl.len().alias('CV-only individuals'))
    .sort('CV-only individuals', descending=True)
    .head(15)
)
occ_dist

### 6.3 Examples — 10 random individuals on each side of the difference

Use `SEED = 42` for reproducibility.

In [ ]:
SEED = 42
N_EX = 10

ex_cu = (
    cultura_only_df.sample(n=min(N_EX, cultura_only_df.height), seed=SEED)
    .select([
        pl.col('wikidata_id').alias('qid'),
        pl.col('name_en').alias('name'),
        pl.col('fy').alias('floruit_year'),
        pl.col('century'),
    ])
    .sort('floruit_year')
)

ex_cv = (
    cv_only_df.sample(n=min(N_EX, cv_only_df.height), seed=SEED)
    .select([
        pl.col('wikidata_code').alias('qid'),
        pl.col('name'),
        pl.col('fy').alias('floruit_year'),
        pl.col('century'),
        pl.col('level1_main_occ'),
        pl.col('level3_main_occ'),
        pl.col('citizenship_1_b'),
        pl.col('citizenship_2_b'),
    ])
    .sort('floruit_year')
)

print('=== Cultura-only — 10 examples ===')
print(ex_cu)
print('\n=== Cross-Verified-only — 10 examples ===')
print(ex_cv)

## 7. Per-century counts

In [ ]:
centuries = np.arange(CENTURY_MIN, CENTURY_MAX + 1)

def per_century(df_pl):
    counts = df_pl.group_by('century').agg(pl.len().alias('n'))
    by_c = dict(counts.iter_rows())
    return np.array([by_c.get(int(c), 0) for c in centuries])

by_c_cultura = per_century(cultura)
by_c_cv      = per_century(cv)

trends = pl.DataFrame({
    'century':  centuries.tolist(),
    'cultura':  by_c_cultura.tolist(),
    'cv':       by_c_cv.tolist(),
}).with_columns(
    (pl.col('cultura') / pl.col('cultura').sum()).alias('cultura_share'),
    (pl.col('cv') / pl.col('cv').sum()).alias('cv_share'),
)
trends.tail(15)

## 8. Figure — raw counts per century

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(trends['century'].to_list(), trends['cultura'].to_list(), color=COL_CULTURA, lw=1.6,
        marker='o', ms=3.5, label=f'Cultura  (N = {n_cultura:,})')
ax.plot(trends['century'].to_list(), trends['cv'].to_list(), color=COL_CV, lw=1.6,
        marker='s', ms=3.5, label=f'Cross-Verified  (N = {n_cv:,})')
ax.set_xlabel('Century (year // 100, BCE ← → CE)')
ax.set_ylabel('Individuals')
ax.set_title('Chinese world — individuals per century', loc='left', pad=12)
ax.legend(loc='upper left', frameon=False)
ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
fig.tight_layout()
plt.show()

## 9. Figure — normalized trends (share of each database's total)

Each series sums to 1.0, so the *shape* of the trajectories can be compared independently of the very different totals.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(trends['century'].to_list(), (trends['cultura_share'] * 100).to_list(), color=COL_CULTURA,
        lw=1.6, marker='o', ms=3.5, label='Cultura')
ax.plot(trends['century'].to_list(), (trends['cv_share'] * 100).to_list(), color=COL_CV,
        lw=1.6, marker='s', ms=3.5, label='Cross-Verified')
ax.set_xlabel('Century (year // 100, BCE ← → CE)')
ax.set_ylabel('Share of database total (%)')
ax.set_title('Chinese world — normalized trends per century', loc='left', pad=12)
ax.legend(loc='upper left', frameon=False)
ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
fig.tight_layout()
plt.show()

## 10. Figure — ratio Cultura / Cross-Verified per century

Where Cultura adds the most coverage relative to Cross-Verified.

In [ ]:
cultura_arr = trends['cultura'].to_numpy().astype(float)
cv_arr      = trends['cv'].to_numpy().astype(float)
ratio = np.divide(cultura_arr, cv_arr, out=np.full_like(cultura_arr, np.nan), where=cv_arr > 0)

fig, ax = plt.subplots(figsize=(11, 4.0))
ax.bar(trends['century'].to_list(), np.nan_to_num(ratio, nan=0.0), color='#7f7f7f', width=0.8)
ax.axhline(1.0, color='black', lw=0.7, ls='--')
ax.set_xlabel('Century (year // 100, BCE ← → CE)')
ax.set_ylabel('Cultura / Cross-Verified')
ax.set_title('Coverage ratio per century', loc='left', pad=12)
ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
fig.tight_layout()
plt.show()

## 11. Export

In [ ]:
trends_path = '../data/chinese_world_cultura_vs_cv_century.csv'
trends.write_csv(trends_path)
print(f'wrote {trends_path}  ({trends.height:,} rows)')

cu_only_out = cultura_only_df.select([
    pl.col('wikidata_id').alias('qid'),
    pl.col('name_en').alias('name'),
    pl.col('fy').alias('floruit_year'),
    pl.col('century'),
])
cu_only_out.write_csv('../data/chinese_world_cultura_only.csv')
print(f'wrote ../data/chinese_world_cultura_only.csv  ({cu_only_out.height:,} rows)')

cv_only_out = cv_only_df.select([
    pl.col('wikidata_code').alias('qid'),
    pl.col('name'),
    pl.col('fy').alias('floruit_year'),
    pl.col('century'),
    pl.col('level1_main_occ'),
    pl.col('level3_main_occ'),
    pl.col('citizenship_1_b'),
    pl.col('citizenship_2_b'),
])
cv_only_out.write_csv('../data/chinese_world_cv_only.csv')
print(f'wrote ../data/chinese_world_cv_only.csv      ({cv_only_out.height:,} rows)')

---

# Part 2 — All worlds: Cultura vs Cross-Verified per century

*(Continuation: extending the per-century comparison from the Chinese world to every world in the database.)*


# World comparisons per century — Cultura vs Cross-Verified

Same logic as `22_chinese_world_cultura_vs_cv_century.ipynb`, applied to five additional cultural worlds:

| World | Cultura side (polities) | Cross-Verified side (citizenships) |
|---|---|---|
| Greek world | Hellenic / Hellenistic / Byzantine polities | derived |
| Muslim world | Caliphates, sultanates, dynasties | derived |
| Japan | Asuka → Empire of Japan | derived |
| Korea | Gojoseon → ROK / DPRK | derived |
| India | Maurya → Republic of India | derived |

The CV citizenship filter for each world is **derived from Cultura's own polity → modern-country mapping** (`polities_modern_countries_cliopatria`). For each world we look up the modern countries linked to its polity list, translate those `country_name` values into Cross-Verified `citizenship_1_b` strings, and use the result as the CV filter. No hand-curated citizenship sets — both sides come from the same Cultura source.

For each world we then compute:
- total individuals in each database;
- counts per century (year `// 100`) using a single floruit-year point estimate;
- normalized share-of-total per century (so the *shape* of trajectories is comparable);
- the Cultura/CV ratio per century.

## 1. Configuration

In [ ]:
import duckdb
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

DB_PATH = '../data/humans_clean.duckdb'
CV_PATH = '../data/similar_databases/cross-verified-database/cross-verified-database.utf8.csv.gz'

CENTURY_MIN, CENTURY_MAX = -16, 20

COL_CULTURA = '#2f5b8a'
COL_CV      = '#b5542a'

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 11,
    'font.family': 'DejaVu Sans',
})

## 2. World definitions (polities only)

Each world is just a list of Cliopatria polity names. The CV citizenship set is derived in the next section.

In [2]:
WORLDS = {
    'Greek world': [
        'Greek City-States', 'Greek Colonies', 'Greek Dark Ages',
        'Athenian Coalition', 'Second Athenian League',
        'Antigonid Dynasty', 'Antigonid Macedonia', 'Macedonian Empire',
        'Ptolemaic Kingdom', 'Seleucid Empire',
        'Achaean League', 'Despotate of Epirus',
        'Duchy of Athens', 'Principality of Achaea',
        'Byzantine Empire', 'Indo-Greeks',
    ],
    'Muslim world': [
        'Rashidun Caliphate', 'Umayyad Caliphate', 'Abbasid Caliphate',
        'Caliphate of Córdoba', 'Fatimid Caliphate', 'Almohad Caliphate',
        'Sokoto Caliphate',
        'Ayyubid Sultanate', 'Mamluk Sultanate', 'Mamluk Dynasty',
        'Almoravid Dynasty', 'Idrisids', 'Aghlabid Dynasty',
        'Tahirid Sultanate', 'Saffarid Dynasty', 'Samanid Empire',
        'Buyid Dynasty', 'Ghaznavid Empire', 'Great Seljuk Empire',
        'Seljuk Dynasty', 'Sultanate of Rum', 'Ilkhanate',
        'Khwarezmid Empire', 'Khwarezmid Dynasty', 'Timurid Empire',
        'Jalayirid Sultanate', 'Safavid Dynasty',
        'Ottoman Empire', 'Ottoman Tripolitania',
        'Hafsid Dynasty', 'Marinid Sultanate', 'Wattasid dynasty',
        'Saadi Sultanate', 'Dabuyid Dynasty', 'Zaydi Alid dynasties',
        'Adal Sultanate', 'Funj Sultanate',
        'Sultanate of Darfur', 'Sultanate of Harar', 'Sultanate of Hobyo',
        'Sultanate of Zanzibar', 'Sultanate of Oman',
        'Sultanate of Muscat and Oman', 'Majeerteen Sultanate',
        'Ifat Sultanate',
        'Delhi Sultanate', 'Bahmani Sultanate',
        'Ahmadnagar Sultanate', 'Berar Sultanate', 'Bidar Sultanate',
        'Bijapur Sultanate', 'Golconda Sultanate',
        'Carnatic Sultanate', 'Jaunpur Sultanate', 'Kashmiri Sultanate',
        'Sultanate of Bengal', 'Mughal Empire',
        'Mataram Sultanate', 'Demak Sultanate',
        'Sultanate of Aceh', 'Sultanate of Banjar', 'Sultanate of Banten',
        'Sultanate of Bima', 'Sultanate of Bone',
        'Sultanate of Brunei', 'Sultanate of Cirebon',
        'Sultanate of Gowa', 'Sultanate of Jambi', 'Sultanate of Johor',
        'Sultanate of Kedah', 'Sultanate of Kutai Kartanegara',
        'Sultanate of Maguindanao', 'Sultanate of Malacca',
        'Sultanate of Mempawah', 'Sultanate of Palembang',
        'Sultanate of Perak', 'Sultanate of Sambaliung',
        'Sultanate of Sambas', 'Sultanate of Selangor',
        'Sultanate of Sukadana', 'Sultanate of Sulu',
        'Sultanate of Terengganu', 'Pontianak Kadriyah Sultanate',
        'Riau-Lingga Sultanate', 'Samudera Pasai Sultanate',
        'Pahang Sultanate',
        'Islamic Republic of Iran', 'Islamic Republic of Pakistan',
    ],
    'Japan': [
        'Asuka Japan', 'Nara Japan', 'Heian Japan',
        'Kamakura Shogunate', 'Ashikaga Shogunate',
        'Warring States Japan', 'Tokugawa Shogunate',
        'Empire of Japan', 'Japan',
    ],
    'Korea': [
        'Gojoseon', 'Goguryeo', 'Baekje', 'Silla', 'Unified Silla',
        'Balhae', 'Hubaekje', 'Goryeo', 'Joseon',
        'Korean Empire', 'Korean Jin',
        'Republic of Korea', "Democratic People's Republic of Korea",
        'Allegiance of Goryeo to Great Jin',
        'Allegiance of Joseon to Ming Dynasty',
    ],
    'India': [
        'Maurya Empire', 'Gupta Empire',
        'Magadha - Haryanka dynasty', 'Magadha - Shaishunaga dynasty',
        'Janapada of Magadha', 'Janapada of Surasena',
        'Kushan Empire', 'Western Kushans', 'Eastern Kushans',
        'Satavahana Dynasty',
        'Late Pallava Empire',
        'Early Cholas', 'Chola Empire',
        'Pandya Dynasty', 'Pandya Empire', 'Early Pandyas',
        'Chalukya Dynasty', 'Western Chalukya Empire',
        'Rashtrakuta Dynasty', 'Pala Empire', 'Sena Dynasty',
        'Hoysala Kingdom', 'Kakatiya Dynasty',
        'Vijayanagara Empire',
        'Maratha Empire', 'Thanjavur Maratha Kingdom',
        'Sikh Empire', 'Sikh Confederacy', 'Sikhs',
        'Gurjara-Pratihara Dynasty', 'Utpala Dynasty',
        'Mughal Empire', 'Republic of India',
        'Alliance between Gupta Empire and Vakataka Kingdom',
        'Vassalage of Jaffna kingdom to Vijayanagara Empire',
    ],
}

for w, ps in WORLDS.items():
    print(f'{w:<15s}  polities = {len(ps):>3d}')

Greek world      polities =  16
Muslim world     polities =  88
Japan            polities =   9
Korea            polities =  15
India            polities =  35


## 3. Derive CV citizenship sets from Cultura's polity → modern-country mapping

`polities_modern_countries_cliopatria` ties each polity to the modern country (or countries) that occupy its former territory, sourced from Wikidata `P36`/`P17`/`P1366`. We:

1. Look up the distinct `country_name` values associated with each world's polity list.
2. Translate those to Cross-Verified `citizenship_1_b` form: spaces → underscores, plus a few aliases where the two databases disagree on a country's canonical name (e.g. `People's Republic of China` → `China`, `Republic of Korea` → `South_Korea`, `United States` → `US`).
3. Keep only translations that actually appear in CV's citizenship vocabulary.

This makes the CV side an automatic shadow of the Cultura polity list — no hand-curated country sets.

In [ ]:
CULTURA_TO_CV_ALIAS = {
    "People's Republic of China": 'China',
    'Republic of Korea':           'South_Korea',
    'South Korea':                 'South_Korea',
    'North Korea':                 'North_Korea',
    'United States':               'US',
    'Kingdom of the Netherlands':  'Netherlands',
    'Czechoslovakia':              'Czechoslovakia',
    'Soviet Union':                'Russia',
    'German Democratic Republic':  'Germany',
    'Socialist Federal Republic of Yugoslavia': 'Yugoslavia',
    'The Bahamas':                 'The_Bahamas',
    'The Gambia':                  'Gambia',
    'Timor-Leste':                 'East_Timor',
    'North Macedonia':             'Macedonia',
}


def cultura_country_to_cv(name):
    if name is None:
        return None
    if name in CULTURA_TO_CV_ALIAS:
        return CULTURA_TO_CV_ALIAS[name]
    return name.replace(' ', '_')


def derive_citizenships(conn, polities, vocab):
    """Return (citizenship_set, country_table) for a polity list."""
    placeholders = ','.join(['?'] * len(polities))
    rows = conn.execute(f"""
        SELECT polity_name, country_name
        FROM polities_modern_countries_cliopatria
        WHERE polity_name IN ({placeholders})
          AND country_name IS NOT NULL
        """, polities).pl().with_columns(
        pl.col('country_name').map_elements(cultura_country_to_cv, return_dtype=pl.Utf8).alias('cv_citizenship'),
    ).with_columns(
        pl.col('cv_citizenship').is_in(vocab).alias('in_cv_vocab'),
    )
    citizenships = set(rows.filter(pl.col('in_cv_vocab'))['cv_citizenship'].to_list())
    return citizenships, rows

## 4. Helpers — load each world from each database

**Cultura.** Filter `individuals_cliopatria` by polity_name (semicolon list), join `individuals_floruit_period` for `floruit_year`. Each individual counted once.

**Cross-Verified.** Filter on `citizenship_1_b` ∪ `citizenship_2_b` against the *derived* citizenship set. Floruit year = midpoint(birth, death) if both present; else `birth + 30` or `death − 30`.

In [ ]:
def load_cultura(conn, polities):
    like_clauses = ' OR '.join(["';' || ic.polity_name || ';' LIKE ?"] * len(polities))
    params = [f'%;{p};%' for p in polities]
    df = conn.execute(f"""
        SELECT DISTINCT ic.wikidata_id,
               fp.floruit_year,
               fp.floruit_period_start AS s,
               fp.floruit_period_end   AS e
        FROM individuals_cliopatria ic
        JOIN individuals_floruit_period fp USING (wikidata_id)
        WHERE ({like_clauses})
    """, params).pl().unique(subset=['wikidata_id'])

    df = df.with_columns(
        pl.coalesce(
            pl.col('floruit_year'),
            ((pl.col('s') + pl.col('e')) / 2).round(0).cast(pl.Int64),
        ).alias('fy')
    ).filter(pl.col('fy').is_not_null()).with_columns(
        pl.col('fy').cast(pl.Int64),
        (pl.col('fy') // 100).cast(pl.Int64).alias('century'),
    )
    return df.select(['wikidata_id', 'fy', 'century'])


def cv_filter(cv_full, citizenships):
    df = cv_full.filter(
        pl.col('citizenship_1_b').is_in(citizenships)
        | pl.col('citizenship_2_b').is_in(citizenships)
    ).with_columns(
        pl.when(pl.col('birth').is_not_null() & pl.col('death_eff').is_not_null())
          .then((pl.col('birth') + pl.col('death_eff')) / 2)
        .when(pl.col('birth').is_not_null())
          .then(pl.col('birth') + 30)
        .when(pl.col('death_eff').is_not_null())
          .then(pl.col('death_eff') - 30)
        .otherwise(None)
        .alias('fy')
    ).filter(pl.col('fy').is_not_null()).unique(subset=['wikidata_code']).with_columns(
        pl.col('fy').round(0).cast(pl.Int64),
    ).with_columns(
        (pl.col('fy') // 100).cast(pl.Int64).alias('century'),
    )
    return df.select(['wikidata_code', 'fy', 'century'])

In [ ]:
# Load CV once and reuse for every world
cv_full = pl.read_csv(
    CV_PATH,
    columns=['wikidata_code', 'citizenship_1_b', 'citizenship_2_b',
             'birth', 'death', 'updated_death_date'],
).with_columns(
    pl.coalesce([pl.col('death'), pl.col('updated_death_date')]).alias('death_eff'),
)
print(f'CV rows loaded: {cv_full.height:,}')

## 5. Compute per-world series

For each world we (a) derive the CV citizenship set from Cultura's polity → country mapping, then (b) load both database series.

In [ ]:
centuries = np.arange(CENTURY_MIN, CENTURY_MAX + 1)

def per_century(df_pl):
    counts = df_pl.group_by('century').agg(pl.len().alias('n'))
    by_c = dict(counts.iter_rows())
    return np.array([by_c.get(int(c), 0) for c in centuries])

# Rebuild CV vocab now that cv_full is loaded
cv_citizenship_vocab = (
    set(cv_full.filter(pl.col('citizenship_1_b').is_not_null())['citizenship_1_b'].to_list())
    | set(cv_full.filter(pl.col('citizenship_2_b').is_not_null())['citizenship_2_b'].to_list())
)

results = {}
country_tables = {}
with duckdb.connect(DB_PATH, read_only=True) as conn:
    for world, polities in WORLDS.items():
        citizenships, ctab = derive_citizenships(conn, polities, cv_citizenship_vocab)
        country_tables[world] = ctab

        cu = load_cultura(conn, polities)
        cv = cv_filter(cv_full, citizenships)
        ser_cu = per_century(cu)
        ser_cv = per_century(cv)
        overlap = len(
            set(cu['wikidata_id'].to_list())
            & set(cv.filter(pl.col('wikidata_code').is_not_null())['wikidata_code'].cast(pl.Utf8).to_list())
        )
        results[world] = {
            'cultura_n':      cu.height,
            'cv_n':           cv.height,
            'overlap':        overlap,
            'citizenships':   citizenships,
            'cultura_series': ser_cu,
            'cv_series':      ser_cv,
        }
        print(f'{world:<15s}  Cultura = {cu.height:>7,}  |  CV = {cv.height:>7,}  |  '
              f'overlap = {overlap:>5,}  |  citizenships = {sorted(citizenships)}')

## 6. Summary table

In [ ]:
summary = pl.DataFrame([
    {
        'World': w,
        'Cultura individuals': r['cultura_n'],
        'Cross-Verified individuals': r['cv_n'],
        'Q-id overlap': r['overlap'],
        'Cultura / CV': round(r['cultura_n'] / r['cv_n'], 2) if r['cv_n'] else None,
        'CV citizenships (derived)': ', '.join(sorted(r['citizenships'])) or '(none)',
    }
    for w, r in results.items()
])
summary

## 7. Per-world figures — raw counts, normalized share, ratio

Three rows per world: raw counts (left), share-of-total (middle), Cultura/CV ratio (right).

In [ ]:
n_worlds = len(WORLDS)
fig, axes = plt.subplots(n_worlds, 3, figsize=(15, 3.2 * n_worlds))
if n_worlds == 1:
    axes = axes[None, :]

for i, (world, r) in enumerate(results.items()):
    cu = r['cultura_series'].astype(float)
    cv = r['cv_series'].astype(float)

    cu_share = cu / cu.sum() if cu.sum() else cu
    cv_share = cv / cv.sum() if cv.sum() else cv
    cv_safe = np.where(cv > 0, cv, np.nan)
    ratio = cu / cv_safe

    # Raw counts
    ax = axes[i, 0]
    ax.plot(centuries, cu, color=COL_CULTURA, lw=1.4, marker='o', ms=3,
            label=f'Cultura  N={r["cultura_n"]:,}')
    ax.plot(centuries, cv, color=COL_CV, lw=1.4, marker='s', ms=3,
            label=f'CV  N={r["cv_n"]:,}')
    ax.set_title(f'{world} — raw counts', loc='left', fontsize=12)
    ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
    ax.set_ylabel('Individuals')
    ax.legend(loc='upper left', frameon=False, fontsize=9)

    # Normalized share
    ax = axes[i, 1]
    ax.plot(centuries, cu_share * 100, color=COL_CULTURA, lw=1.4, marker='o', ms=3, label='Cultura')
    ax.plot(centuries, cv_share * 100, color=COL_CV, lw=1.4, marker='s', ms=3, label='CV')
    ax.set_title(f'{world} — share of total (%)', loc='left', fontsize=12)
    ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
    ax.set_ylabel('% of database total')
    ax.legend(loc='upper left', frameon=False, fontsize=9)

    # Cultura / CV ratio
    ax = axes[i, 2]
    ax.bar(centuries, np.nan_to_num(ratio, nan=0.0), color='#7f7f7f', width=0.85)
    ax.axhline(1.0, color='black', lw=0.7, ls='--')
    ax.set_title(f'{world} — Cultura / CV ratio', loc='left', fontsize=12)
    ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
    ax.set_ylabel('ratio')
    if i == n_worlds - 1:
        for col in range(3):
            axes[i, col].set_xlabel('Century (year // 100, BCE ← → CE)')

fig.tight_layout()
plt.show()

## 8. Combined normalized trends — all worlds on one axis

Comparing the temporal *shape* of coverage across worlds, each as a share of its own database total.

In [ ]:
fig, (axa, axb) = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
palette = ['#2f5b8a', '#b5542a', '#6a9e3a', '#7f3f8a', '#c89a3a']

for color, (world, r) in zip(palette, results.items()):
    cu = r['cultura_series'].astype(float)
    cv = r['cv_series'].astype(float)
    cu_share = cu / cu.sum() * 100 if cu.sum() else cu
    cv_share = cv / cv.sum() * 100 if cv.sum() else cv
    axa.plot(centuries, cu_share, color=color, lw=1.5, marker='o', ms=3, label=world)
    axb.plot(centuries, cv_share, color=color, lw=1.5, marker='o', ms=3, label=world)

for ax, name in [(axa, 'Cultura'), (axb, 'Cross-Verified')]:
    ax.set_title(f'{name} — share per century (%)', loc='left', pad=10)
    ax.set_xlabel('Century (year // 100, BCE ← → CE)')
    ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
    ax.legend(loc='upper left', frameon=False, fontsize=9)
axa.set_ylabel('% of database total')
fig.tight_layout()
plt.show()

## 9. Export per-century tables and the per-world country mappings

In [ ]:
rows = []
for world, r in results.items():
    cu = r['cultura_series'].astype(int)
    cv = r['cv_series'].astype(int)
    cu_total = int(cu.sum()) or 1
    cv_total = int(cv.sum()) or 1
    for ci, c in enumerate(centuries):
        rows.append({
            'world':         world,
            'century':       int(c),
            'cultura':       int(cu[ci]),
            'cv':            int(cv[ci]),
            'cultura_share': float(cu[ci]) / cu_total,
            'cv_share':      float(cv[ci]) / cv_total,
        })

trends = pl.DataFrame(rows)
trends_path = '../data/worlds_cultura_vs_cv_century.csv'
trends.write_csv(trends_path)
print(f'wrote {trends_path}  ({trends.height:,} rows)')

mapping_rows = []
for world, ctab in country_tables.items():
    for r in ctab.iter_rows(named=True):
        mapping_rows.append({
            'world':          world,
            'polity_name':    r['polity_name'],
            'country_name':   r['country_name'],
            'cv_citizenship': r['cv_citizenship'],
            'in_cv_vocab':    bool(r['in_cv_vocab']),
        })
mapping_df = pl.DataFrame(mapping_rows)
mapping_path = '../data/worlds_polity_country_mapping.csv'
mapping_df.write_csv(mapping_path)
print(f'wrote {mapping_path}  ({mapping_df.height:,} rows)')